In [2]:
import torch
from transformers import BertForQuestionAnswering
from transformers import BertTokenizer
from transformers import pipeline
import pandas as pd
from google.colab import drive
from datasets import Dataset
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Import previously translated data sets
dataset_tr = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/NLP/train_translated.csv")
dataset_val = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/NLP/validation_translated.csv")

In [4]:
#Model
model = BertForQuestionAnswering.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')

#Tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [5]:
question = dataset_tr.iloc[1]["question_translated"]

context = dataset_tr.iloc[1]["context"]

encoding = tokenizer.encode_plus(text=question,text_pair=context)

inputs = encoding['input_ids']  #Token embeddings
sentence_embedding = encoding['token_type_ids']  #Segment embeddings
tokens = tokenizer.convert_ids_to_tokens(inputs) #input tokens


In [6]:
output = model(input_ids=torch.tensor([inputs]), token_type_ids=torch.tensor([sentence_embedding]))
start_index = torch.argmax(output.start_logits)
end_index = torch.argmax(output.end_logits)

In [10]:
answer = ' '.join(tokens[start_index:end_index+1])

In [11]:
answer

'wilhelm ron ##t ##gen'

##**Trainable Model**

In [7]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-large-uncased-whole-word-masking-finetuned-squad")

**Trainings data prepocessing**

In [8]:
# Setting up
questions = [q.strip() for q in dataset_tr["question"]]
context = [q.strip() for q in dataset_tr["context"]]
answer_start = [q for q in dataset_tr["answer_start"]]
answer = [q for q in dataset_tr["answer"]]

inputs = tokenizer(
        questions,
        context,
        max_length=384,
        truncation="only_second",
        return_offsets_mapping=True,
        padding="max_length",
    )

offset_mapping = inputs.pop("offset_mapping")

In [9]:
# Preprocessing trainings data
start_pos = []
end_pos = []
for i, offset in enumerate(offset_mapping):
    start_c = answer_start[i]
    end_c = start_c + len(answer[i])

    # Find the start and end of the context
    sequence_ids = inputs.sequence_ids(i)

    # Find the start and end of the context
    idx = 0
    while sequence_ids[idx] != 1:
        idx += 1
    start_context = idx
    while sequence_ids[idx] == 1:
        idx += 1
    end_context = idx - 1

    # If the answer is not fully inside the context, label it (0, 0)
    if offset[start_context][0] > end_c or offset[end_context][1] < start_c:
        start_pos.append(0)
        end_pos.append(0)
    else:
        # Otherwise it's the start and end token positions
        idx = start_context
        while idx <= end_context and offset[idx][0] <= start_c:
            idx += 1
        start_pos.append(idx - 1)

        idx = end_context
        while idx >= start_context and offset[idx][1] >= end_c:
            idx -= 1
        end_pos.append(idx + 1)

dataset_tr["start_positions"] = start_pos
dataset_tr["end_positions"] = end_pos

data = {'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask'],
        'start_positions':start_pos,
        'end_positions': end_pos,
       }
df = pd.DataFrame(data)
df.to_csv('encoding_train.csv',index=False)
train = Dataset.from_pandas(df)

**Test data prepocessing**

In [10]:
# Setting up
questions = [q.strip() for q in dataset_val["question"]]
context = [q.strip() for q in dataset_val["context"]]
answer_start = [q for q in dataset_val["answer_start"]]
answer = [q for q in dataset_val["answer"]]

inputs = tokenizer(
        questions,
        context,
        max_length=512,
        truncation="only_second",
        return_offsets_mapping=True,
        padding="max_length",
    )

offset_mapping = inputs.pop("offset_mapping")

In [11]:
# Preprocessing trainings data
start_pos = []
end_pos = []
for i, offset in enumerate(offset_mapping):
    start_c = answer_start[i]
    end_c = start_c + len(answer[i])

    # Find the start and end of the context
    sequence_ids = inputs.sequence_ids(i)

    # Find the start and end of the context
    idx = 0
    while sequence_ids[idx] != 1:
        idx += 1
    start_context = idx
    while sequence_ids[idx] == 1:
        idx += 1
    end_context = idx - 1

    # If the answer is not fully inside the context, label it (0, 0)
    if offset[start_context][0] > end_c or offset[end_context][1] < start_c:
        start_pos.append(0)
        end_pos.append(0)
    else:
        # Otherwise it's the start and end token positions
        idx = start_context
        while idx <= end_context and offset[idx][0] <= start_c:
            idx += 1
        start_pos.append(idx - 1)

        idx = end_context
        while idx >= start_context and offset[idx][1] >= end_c:
            idx -= 1
        end_pos.append(idx + 1)

dataset_val["start_positions"] = start_pos
dataset_val["end_positions"] = end_pos

data = {'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask'],
        'start_positions':start_pos,
        'end_positions': end_pos,
       }
df = pd.DataFrame(data)
df.to_csv('encoding_test.csv',index=False)
test = Dataset.from_pandas(df)

In [ ]:
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer
model = AutoModelForQuestionAnswering.from_pretrained("bert-large-uncased-whole-word-masking-finetuned-squad")

from transformers import DefaultDataCollator

data_collator = DefaultDataCollator()

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.1,
    report_to="none",
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,                 # evaluate less frequently than logging
    save_strategy="steps",
    save_steps=200,                 # save model every eval
    load_best_model_at_end=True,    # restore best validation checkpoint
    metric_for_best_model="loss",   # pick metric to judge best model
    greater_is_better=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train,
    eval_dataset=test,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
/tmp/ipython-input-1359979179.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss
200,2.288100,2.642035
400,2.028800,2.395744
600,2.020000,2.281075
800,1.777500,2.123605
1000,1.182000,2.388843
1200,1.325300,2.283422
1400,1.190400,2.387391
1600,1.074600,2.423265
1800,0.521500,3.012017
2000,0.604100,2.811271


TrainOutput(global_step=2376, training_loss=1.3136715294937493, metrics={'train_runtime': 1171.0022, 'train_samples_per_second': 16.23, 'train_steps_per_second': 2.029, 'total_flos': 1.323755728904448e+16, 'train_loss': 1.3136715294937493, 'epoch': 3.0})

In [10]:
metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    accuracy = metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    return {"accuracy": accuracy["accuracy"], "f1": f1["f1"]}


eval_args = TrainingArguments(
    output_dir=f"./eval",
    per_device_eval_batch_size=8,
    do_train=False,
    do_eval=True,
    logging_dir=f"./logs",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=eval_args,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

eval_result = trainer.evaluate(eval_dataset=test)
print(f"Validation results for {lang}: {eval_result}")

NameError: name 'evaluate' is not defined